# Brute-Force Pairs Finder
This notebook implements the "brute-force" approach to finding tradable pairs. The process is:

Define a Universe: We'll select a group of related stocks (e.g., the S&P 500 Financials) to test.
Get Data: We'll use our DataHandler to fetch 3-5 years of daily price data for this universe.
Pair Up: We'll programmatically create every possible unique pair from this universe.
Test for Cointegration: For each pair, we will:
Run an OLS regression to find the hedge ratio (k).
Calculate the "spread" (the residuals from the regression).
Run the Augmented Dickey-Fuller (ADF) test on the spread to see if it's stationary.
Save Results: We'll save all pairs that pass the test (p-value < 0.05) to a list.

In [5]:
import pandas as pd
import numpy as np
import os
import itertools
from dotenv import load_dotenv

# --- Math & Stats Libraries ---
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

# --- Alpaca & Data Handling ---
# We'll use the DataHandler class we built, assuming it's in the 'src' folder
# If running this from the 'notebooks' folder, we need to adjust the path
import sys
sys.path.append('../src') # This allows us to import from the 'src' folder
from data_handler import DataHandler

print("Libraries imported successfully.")

ImportError: cannot import name 'DataHandler' from 'data_handler' (c:\Users\trash\trading-bot\notebooks\../src\data_handler.py)

In [ ]:
# --- Step 1: Define Universe & Timeframe ---

# A targeted "brute-force" is better than testing thousands of stocks.
# Let's use a list of major US financial stocks (components of the XLF ETF).
# This provides a good fundamental reason for cointegration.
UNIVERSE = [
    'JPM', 'BAC', 'WFC', 'MS', 'GS', 'C', 'BLK', 'SPGI', 'AXP',
    'SCHW', 'CB', 'PGR', 'MET', 'AIG', 'TRV', 'MMC', 'AON',
    'COF', 'USB', 'MCO', 'PYPL', 'V', 'MA'
]

# Cointegration is a long-term relationship, so we need several years of data.
START_DATE = '2020-01-01'
END_DATE = '2024-01-01' # Test on data up to the start of 2024
TIMEFRAME = '1D' # Daily data is standard for this

print(f"Testing {len(UNIVERSE)} stocks from {START_DATE} to {END_DATE}.")

In [ ]:
# --- Step 2: Get & Prepare Price Data ---

print("Fetching historical data...")
dh = DataHandler(paper_trading=True)

# Get the dictionary of DataFrames from our handler
hist_data = dh.get_historical_bars(UNIVERSE, TIMEFRAME, START_DATE, END_DATE)

# We need to combine this into a single DataFrame of closing prices
print("Processing data into a master DataFrame...")
close_prices = pd.DataFrame()

for symbol in UNIVERSE:
    if hist_data and symbol in hist_data and not hist_data[symbol].empty:
        # We need to make sure the index is a DatetimeIndex
        df = hist_data[symbol]
        df.index = pd.to_datetime(df.index)
        
        # Get just the closing prices
        close_prices[symbol] = df['close']
    else:
        print(f"Warning: No data for {symbol}. It will be skipped.")

# Drop any rows with missing data for any stock
close_prices.dropna(inplace=True)

# Re-update our universe list to only include stocks we have data for
UNIVERSE = close_prices.columns.tolist()

print(f"Data prepared. Master DataFrame shape: {close_prices.shape}")
display(close_prices.head())

In [ ]:
# --- Step 3: Cointegration Test Function ---

def find_cointegration(series_1, series_2):
    """
    Tests for cointegration between two price series.
    
    1. Runs OLS regression: series_1 = k * series_2 + intercept
    2. Calculates the spread (residuals).
    3. Runs ADF test on the spread.
    
    :return: (hedge_ratio, adf_p_value)
    """
    
    # 1. Run OLS regression
    # We add a constant (intercept) to the independent variable
    series_2_with_const = sm.add_constant(series_2)
    model = sm.OLS(series_1, series_2_with_const)
    results = model.fit()
    
    hedge_ratio = results.params[1] # 'k'
    
    # 2. Calculate the spread
    spread = series_1 - hedge_ratio * series_2
    
    # 3. Run ADF test on the spread
    # The null hypothesis of ADF is that the series IS non-stationary
    # We want a low p-value to reject the null hypothesis
    adf_test = adfuller(spread)
    adf_p_value = adf_test[1] # The p-value
    
    return hedge_ratio, adf_p_value

In [ ]:
# --- Step 4: The Brute-Force Loop ---

print("Running cointegration tests on all pairs...")

# Set our significance threshold
P_VALUE_THRESHOLD = 0.05

# Create all unique pairs of stocks
all_pairs = list(itertools.combinations(UNIVERSE, 2))

cointegrated_pairs = []

for symbol_a, symbol_b in all_pairs:
    
    series_a = close_prices[symbol_a]
    series_b = close_prices[symbol_b]
    
    hedge_ratio, p_value = find_cointegration(series_a, series_b)
    
    if p_value < P_VALUE_THRESHOLD:
        print(f"Found Cointegrated Pair: {symbol_a} / {symbol_b} | p-value: {p_value:.4f} | hedge_ratio: {hedge_ratio:.2f}")
        cointegrated_pairs.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'hedge_ratio': hedge_ratio,
            'p_value': p_value
        })

print("\n--- Test Complete ---")
print(f"Total pairs tested: {len(all_pairs)}")
print(f"Total cointegrated pairs found: {len(cointegrated_pairs)}")

In [ ]:
# --- Step 5: Analyze Results ---

# Convert the results into a clean DataFrame for analysis
results_df = pd.DataFrame(cointegrated_pairs)

print("\nCointegrated Pairs Summary:")

display(results_df)

# You can now save this to a file
# results_df.to_csv('cointegrated_pairs.csv', index=False)